# Python Notebook 3: Regression DT & Ensemble Learners
**Author:** Mihiri Pabasara | **Student ID:** [Your Student ID]
**Module:** 5DATA002W.2 – Machine Learning & Data Mining
**Peer Reviewer:** [Peer Reviewer Full Name] | **Review Date:** [Date of Review]

> Code reused from **Seminar 6 (Decision Trees)**, **Seminar 7 (Ensemble Learning)**, and **Code Reuse Session 3** as noted per cell.

**This notebook contains TWO parts:**

**PART A:** Probability-based Voting Ensemble Classifier for Loan Approval Status
- Base Learners: Naïve Bayes (NB) + Logistic Regression (LR)

**PART B:** Decision Tree Regression for Maximum Loan Amount
- DT-1: Fully Grown Decision Tree Regressor
- DT-2: Pruned Decision Tree Regressor (max_depth=4)

**Target Encoding (Classification):**
- 0 = Approved
- 1 = Rejected

## 1. Import Libraries
**Reused From:** Seminar 6 and Seminar 7 – all required imports

In [ ]:
# Import data manipulation library
import pandas as pd
# Import numpy for numerical calculations
import numpy as np
# Import train-test split and cross-validation utilities
from sklearn.model_selection import train_test_split
# Import Decision Tree algorithms for regression
from sklearn.tree import DecisionTreeRegressor
# Import tree module for visualisation
from sklearn import tree, metrics
# Import matplotlib for plotting
import matplotlib.pyplot as plt
# Import classification evaluation metrics
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)
# Import Voting Ensemble Classifier
from sklearn.ensemble import VotingClassifier
# Import base learners from Notebook 2
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
# Import standard scaler for LR and KNN compatibility
from sklearn.preprocessing import StandardScaler
# Suppress non-critical warnings
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully.")

## 2. Mount Google Drive
**Reused From:** Seminar 1, Part (A) – mounting Google Drive

In [ ]:
from google.colab import drive
# Mount Google Drive to access prepared datasets
drive.mount('/content/drive')

---
## PART A: Voting Ensemble Classifier for Loan Approval Status

## 3. Load Classification Dataset
**Reused From:** Seminar 1 – pd.read_csv()

In [ ]:
# Load the prepared classification dataset (saved from Notebook 1)
df_class = pd.read_csv('/content/drive/MyDrive/loan_classification_data.csv')
# Display shape to verify correct load
print("Classification dataset shape:", df_class.shape)

## 4. Prepare and Split Classification Data
**Reused From:** Seminar 3 and Seminar 7 – feature assignment and train-test split

In [ ]:
# Assign input features (all columns except the target variable)
X_c = df_class.drop(['loan_approval_status'], axis=1)
# Assign the target variable (0=Approved, 1=Rejected)
y_c = df_class['loan_approval_status']

# Scale features with StandardScaler — required for LR and NB compatibility
scaler = StandardScaler()
# Fit and transform on full dataset before split (consistent with how models were built)
X_c_scaled = scaler.fit_transform(X_c)

# Split into 75% training and 25% test — same ratio as Notebook 2 for consistency
# stratify=y_c preserves class ratio; random_state=42 ensures same test instances as Notebook 2
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c_scaled, y_c, test_size=0.25, random_state=42, stratify=y_c)
print(f"Training set: {X_train_c.shape}, Test set: {X_test_c.shape}")
print("Test class distribution (0=Approved, 1=Rejected):")
print(y_test_c.value_counts(normalize=True) * 100)

## 5. Build Base Learners
**Reused From:** Seminar 7 – Building Base Learners

NB and LR are selected as the two base learners because both support `predict_proba` (probability outputs) which is required for soft voting. Both were also among the stronger performers on Recall for the Rejected class in Notebook 2.

In [ ]:
# --- Build Base Learner 1: Naïve Bayes ---
nb = GaussianNB()
# Train NB on the scaled training data
nb.fit(X_train_c, y_train_c)
# Predict on test set
y_pred_nb = nb.predict(X_test_c)
print("NB Classification Report:")
print(classification_report(y_test_c, y_pred_nb, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Confusion matrix for Naive Bayes base learner
nb_cm = confusion_matrix(y_test_c, y_pred_nb, labels=nb.classes_)
# Display confusion matrix with appropriate labels
ConfusionMatrixDisplay(nb_cm, display_labels=['Approved(0)', 'Rejected(1)']).plot(values_format='.0f')
plt.title('Confusion Matrix: NB (Base Learner 1)')
plt.show()
# ROC curve for NB base learner
RocCurveDisplay.from_estimator(nb, X_test_c, y_test_c)
plt.title('ROC Curve: NB (Base Learner 1)')
plt.show()

In [ ]:
# --- Build Base Learner 2: Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
# Train LR on the scaled training data
lr.fit(X_train_c, y_train_c)
# Predict on test set
y_pred_lr = lr.predict(X_test_c)
print("LR Classification Report:")
print(classification_report(y_test_c, y_pred_lr, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Confusion matrix for Logistic Regression base learner
lr_cm = confusion_matrix(y_test_c, y_pred_lr, labels=lr.classes_)
# Display confusion matrix with appropriate labels
ConfusionMatrixDisplay(lr_cm, display_labels=['Approved(0)', 'Rejected(1)']).plot(values_format='.0f')
plt.title('Confusion Matrix: LR (Base Learner 2)')
plt.show()
# ROC curve for LR base learner
RocCurveDisplay.from_estimator(lr, X_test_c, y_test_c)
plt.title('ROC Curve: LR (Base Learner 2)')
plt.show()

## 6. Build Probability-Based Voting Ensemble (Soft Voting)
**Reused From:** Seminar 7 – Voting (Soft Voting) classifier

NB and LR are combined using soft voting (probability averaging). KNN is excluded as it showed lower Recall for the Rejected class compared to NB and LR in Notebook 2. Both NB and LR support `predict_proba`, which is required for soft voting.

In [ ]:
# Import the VotingClassifier from sklearn ensemble
from sklearn.ensemble import VotingClassifier

# Declare both base learners as a list of (name, estimator) tuples
base_learners = [('NB', nb), ('LR', lr)]

# Create a soft-voting ensemble classifier
# voting='soft' averages the predicted class probabilities from NB and LR
ensemble = VotingClassifier(estimators=base_learners, voting='soft')

# Fit the ensemble learner on the training data
ensemble.fit(X_train_c, y_train_c)

# Predict loan approval status on the test data using the ensemble
y_pred_ensemble = ensemble.predict(X_test_c)
print("Ensemble classifier (NB + LR, soft voting) trained and tested.")

In [ ]:
# Confusion matrix for the Voting Ensemble Learner
ens_cm = confusion_matrix(y_test_c, y_pred_ensemble, labels=ensemble.classes_)
# Display ensemble confusion matrix
ConfusionMatrixDisplay(ens_cm, display_labels=['Approved(0)', 'Rejected(1)']).plot(values_format='.0f')
plt.title('Confusion Matrix: Voting Ensemble (NB + LR)')
plt.show()

# Full classification report for the Ensemble Learner
print("Ensemble Learner Classification Report:")
print(classification_report(y_test_c, y_pred_ensemble, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# ROC curve for the Voting Ensemble Learner
RocCurveDisplay.from_estimator(ensemble, X_test_c, y_test_c)
plt.title('ROC Curve: Voting Ensemble (NB + LR)')
plt.show()

## 7. Compare Ensemble vs Base Learners
**Reused From:** Seminar 7 – comparing ensemble performance to individual base learners

In [ ]:
from sklearn.metrics import recall_score, roc_auc_score

# Compare Recall for Rejected class (pos_label=1) across base learners and ensemble
print("Performance Comparison: Base Learners vs Voting Ensemble")
print("-" * 60)
for name, model, y_pred in [('NB (Base 1)', nb, y_pred_nb),
                              ('LR (Base 2)', lr, y_pred_lr),
                              ('Ensemble (NB+LR)', ensemble, y_pred_ensemble)]:
    # Calculate Recall for Rejected class (1=Rejected is the positive class of interest)
    rec = recall_score(y_test_c, y_pred, pos_label=1)
    # Calculate AUC-ROC using predicted probabilities
    auc = roc_auc_score(y_test_c, model.predict_proba(X_test_c)[:, 1])
    print(f"{name:25s} | Recall(Rejected=1): {rec:.4f} | AUC-ROC: {auc:.4f}")

---
## PART B: Decision Tree Regression for Maximum Loan Amount

## 8. Load Regression Dataset
**Reused From:** Seminar 1 – pd.read_csv(); Seminar 6, Part 3 – regression

This dataset contains only **approved** clients (loan_approval_status == 0) and uses `max_allowed_loan` as the target variable.

In [ ]:
# Load the prepared regression dataset (approved clients only, saved from Notebook 1)
df_reg = pd.read_csv('/content/drive/MyDrive/loan_regression_data.csv')
# Display shape and column names
print("Regression dataset shape:", df_reg.shape)
print("Columns:", list(df_reg.columns))
# Summary statistics for the regression target
df_reg.describe().transpose()

## 9. Prepare and Split Regression Data
**Reused From:** Seminar 6, Part 3, Step 2 – define X and y for regression

In [ ]:
# Assign all variables except the regression target to X
X_r = df_reg.drop(['max_allowed_loan'], axis=1)
# Assign max_allowed_loan as the regression target variable
y_r = df_reg['max_allowed_loan']
print("Input features:", list(X_r.columns))
print("Input shape:", X_r.shape)
print(f"Target range: £{y_r.min():,.2f} to £{y_r.max():,.2f}")

# Split into 90% training and 10% test
# Decision Trees do not require scaling; raw feature values are used directly
# random_state=10 ensures reproducibility
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_r, y_r, test_size=0.1, random_state=10)
print(f"Training: {X_train_r.shape}, Test: {X_test_r.shape}")

## 10. DT-1: Fully Grown Decision Tree Regressor
**Reused From:** Seminar 6, Part 3, Step 3 – DecisionTreeRegressor fully grown

In [ ]:
# Instantiate a fully grown Decision Tree Regressor (no depth constraint)
# random_state=42 ensures reproducibility
regressor_DT1 = DecisionTreeRegressor(random_state=42)
# Train the fully grown DT on the raw (unscaled) training data
# Decision Trees are scale-invariant; no normalisation is needed
regressor_DT1.fit(X_train_r, y_train_r)
# Make predictions on the test set
y_pred_DT1 = regressor_DT1.predict(X_test_r)
# Display the maximum depth achieved by the fully grown tree
print("DT-1 (Fully Grown) max depth:", regressor_DT1.tree_.max_depth)

In [ ]:
# Evaluate DT-1 regression performance on the test set
print("=== DT-1 (Fully Grown) Regression Metrics ===")
# Mean Absolute Error: average absolute prediction error in same units (GBP)
print('MAE:  ', metrics.mean_absolute_error(y_test_r, y_pred_DT1))
# Mean Squared Error: penalises large errors more (sensitive to extreme value errors)
print('MSE:  ', metrics.mean_squared_error(y_test_r, y_pred_DT1))
# Root MSE: interpretable in original units (GBP)
print('RMSE: ', np.sqrt(metrics.mean_squared_error(y_test_r, y_pred_DT1)))
# R-Squared: proportion of variance in max_allowed_loan explained by the model
print('R2:   ', metrics.r2_score(y_test_r, y_pred_DT1))

## 11. DT-2: Pruned Decision Tree Regressor (max_depth=4)
**Reused From:** Seminar 6, Part 3, Step 4 – pruned DecisionTreeRegressor

**Pruning method used:** Pre-pruning via the `max_depth=4` hyperparameter. This limits the tree to four levels during the growing phase (before the tree becomes fully grown), which reduces overfitting and improves generalisation to unseen clients. Benefits include a simpler, more interpretable tree and lower variance. A disadvantage is that the model may underfit if the underlying relationship in the loan data requires more complex splits than four levels allow.

In [ ]:
# Instantiate a pruned Decision Tree Regressor limited to 4 levels (pre-pruning)
# max_depth=4 restricts tree growth to prevent overfitting on the training data
regressor_DT2 = DecisionTreeRegressor(max_depth=4, random_state=42)
# Train the pruned DT on the raw (unscaled) training data
regressor_DT2.fit(X_train_r, y_train_r)
# Make predictions on the test set
y_pred_DT2 = regressor_DT2.predict(X_test_r)
# Confirm actual depth of pruned tree
print("DT-2 (Pruned, max_depth=4) depth:", regressor_DT2.tree_.max_depth)

In [ ]:
# Evaluate DT-2 regression performance on the test set
print("=== DT-2 (Pruned, max_depth=4) Regression Metrics ===")
# Mean Absolute Error: average absolute prediction error in GBP
print('MAE:  ', metrics.mean_absolute_error(y_test_r, y_pred_DT2))
# Mean Squared Error: penalises large errors more heavily (key metric per success criteria)
print('MSE:  ', metrics.mean_squared_error(y_test_r, y_pred_DT2))
# Root MSE for interpretability in same units (GBP)
print('RMSE: ', np.sqrt(metrics.mean_squared_error(y_test_r, y_pred_DT2)))
# R-Squared: proportion of variance explained
print('R2:   ', metrics.r2_score(y_test_r, y_pred_DT2))

## 12. Visualise DT-1 (Fully Grown)
**Reused From:** Seminar 6, Step 3 – tree.plot_tree()

In [ ]:
# Plot the fully grown decision tree (DT-1) - first 3 levels shown for readability
# The full tree is too large to display clearly; top 3 levels capture the primary splits
DT1_fig = plt.figure(figsize=(200, 100))
tree.plot_tree(regressor_DT1,
               feature_names=list(X_train_r.columns),
               filled=True,
               fontsize=10)
plt.title('DT-1: Fully Grown Regression Decision Tree', fontsize=20)
# Save as SVG (vector format - scalable without quality loss, ideal for report)
DT1_fig.savefig('DT1_FullyGrown.svg')
plt.show()
print("DT-1 saved as DT1_FullyGrown.svg")

In [ ]:
# Display top 3 levels of DT-1 for readability in report
DT1_top_fig = plt.figure(figsize=(20, 10))
tree.plot_tree(
    regressor_DT1,
    feature_names=list(X_train_r.columns),
    filled=True,
    fontsize=8,
    max_depth=3   # Show only first 3 levels for readability
)
plt.title('DT-1: Fully Grown Regression Decision Tree (Top 3 Levels Shown)', fontsize=16)
plt.show()

## 13. Visualise DT-2 (Pruned)
**Reused From:** Seminar 6, Step 5 – pruned tree.plot_tree()

In [ ]:
# Plot the pruned decision tree (DT-2) - full tree is visible at max_depth=4
DT2_fig = plt.figure(figsize=(50, 30))
tree.plot_tree(regressor_DT2,
               feature_names=list(X_train_r.columns),
               filled=True,
               fontsize=12)
plt.title('DT-2: Pruned Regression Decision Tree (max_depth=4)', fontsize=16)
# Save as PNG (high quality for insertion into analysis report)
DT2_fig.savefig('DT2_Pruned.png', dpi=150, bbox_inches='tight')
plt.show()
print("DT-2 saved as DT2_Pruned.png")

## 14. Predict Maximum Loan Amount for Client 60256
**Reused From:** Seminar 3, Part (2) – predict() on new data

**Client 60256 details (from coursework specification):**

| Variable | Value |
|---|---|
| Age | 56 years |
| Income | £57,000 |
| Home Ownership | Rent → encoded as 3 (alphabetical: MORTGAGE=0, OTHER=1, OWN=2, RENT=3) |
| Employment Length | 15 years |
| Loan Intent | Medical → encoded as 3 (alphabetical: DEBTCONSOLIDATION=0, EDUCATION=1, HOMEIMPROVEMENT=2, MEDICAL=3) |
| Loan Amount | £25,700 |
| Loan Interest Rate | 23% |
| Loan-to-Income Ratio | 10% = 0.10 |
| Payment Default on File | No → encoded as 0 |
| Credit History Length | 35 years |

In [ ]:
# Client 60256 attribute values encoded consistently with Notebook 1 LabelEncoder mapping
# home_ownership: MORTGAGE=0, OTHER=1, OWN=2, RENT=3 (alphabetical order)
# loan_intent: DEBTCONSOLIDATION=0, EDUCATION=1, HOMEIMPROVEMENT=2, MEDICAL=3, PERSONAL=4, VENTURE=5
# payment_default_on_file: N=0, Y=1 (explicitly mapped in Notebook 1)
client_60256 = pd.DataFrame([{
    'age': 56,
    'income': 57000,
    'home_ownership': 3,            # RENT -> 3 (alphabetical encoding)
    'emplyment_length': 15,
    'loan_intent': 3,               # MEDICAL -> 3 (alphabetical encoding)
    'loan_amount': 25700,
    'loan_interest_rate': 23.0,
    'loan_income_ratio': 0.10,      # LTI = 10% expressed as decimal
    'payment_default_on_file': 0,   # No default on file -> 0
    'credit_history_length': 35
}])

# Ensure column order matches training data
client_60256 = client_60256[list(X_train_r.columns)]

# Predict using DT-1 (fully grown)
prediction_DT1 = regressor_DT1.predict(client_60256)
print(f"DT-1 Predicted Maximum Loan Amount for Client 60256: £{prediction_DT1[0]:,.2f}")

# Predict using DT-2 (pruned) for comparison
prediction_DT2 = regressor_DT2.predict(client_60256)
print(f"DT-2 Predicted Maximum Loan Amount for Client 60256: £{prediction_DT2[0]:,.2f}")

In [ ]:
# Show the decision path for Client 60256 through DT-2 (pruned - more interpretable)
# This trace explains HOW the model arrived at its prediction
decision_path = regressor_DT2.decision_path(client_60256)
feature_names = list(X_train_r.columns)
threshold = regressor_DT2.tree_.threshold
feature = regressor_DT2.tree_.feature

print("Decision path for Client 60256 through DT-2 (Pruned):")
node_indicator = decision_path.toarray()[0]
node_ids = np.where(node_indicator)[0]
# Loop through each node in the decision path and display the split condition
for node_id in node_ids:
    if feature[node_id] != -2:  # -2 indicates a leaf node (end of path)
        fname = feature_names[feature[node_id]]
        thr = threshold[node_id]
        val = client_60256.iloc[0][fname]
        direction = "<=" if val <= thr else ">"
        print(f"  Node {node_id}: {fname} = {val:.2f} {direction} {thr:.2f}")
print(f"  -> Final predicted maximum loan amount: £{prediction_DT2[0]:,.2f}")